### RFE and Mutual Information to choose optimal variable for the dataset (and study potentially redundant ones)

In [1]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.feature_selection import RFE, mutual_info_regression, mutual_info_classif
from sklearn.model_selection import train_test_split
import seaborn as sns

# load data
df = pd.read_csv("../houses_fixed.csv")

# handle these better with your own dataset, here we are simply dropping these
df = df.drop(['id', 'date', 'zipcode', 'lat', 'long'], axis=1)

# change this to your own target variable
TARGET_VARIABLE = "price"

### RFE - recursive feature elimination

In [2]:
# typical X/y -split
X = df.drop(TARGET_VARIABLE, axis=1)
y = df[TARGET_VARIABLE]

# define model (linear regression, random forest, XGBoost etc.)
# technically you can use pretty much any classic ML algorithm
# model = LinearRegression()

# idea: you might want to try RFE with multuple ML algorithms
# and then cross-validate the results (which results are common in all models)

# another idea: if you plan on using e.g. XGBoost in your final model
# it's probably a good idea to use the same XGBoost here
model = RandomForestRegressor()

# create RFE, place the model and choose number of optimal variables
rfe = RFE(estimator=model, n_features_to_select=7)

# fit the RFE model with our data
rfe.fit(X, y)

# get rankings and results
rankings = rfe.ranking_
support = rfe.support_

# build a DataFrame to wrap up result for easier inspection
results_df = pd.DataFrame({
    "Feature": X.columns,
    "Ranking": rankings,
    "Selected": support
}).sort_values(by="Ranking")

# you can use these results with any other knowledge you have
# from other optimal variable selection tools, and cross-validation
results_df

,Feature,Ranking,Selected
3,sqft_lot,1,True
2,sqft_living,1,True
9,sqft_above,1,True
14,sqft_lot15,1,True
13,sqft_living15,1,True
11,yr_built,1,True
8,grade,1,True
5,waterfront,2,False
1,bathrooms,3,False
6,view,4,False


In [3]:
# the selection loosely follow the correlations it seems
# this might imply the data is more or less linear or something else
df.corr()[TARGET_VARIABLE].sort_values(ascending=False)

price            1.000000
sqft_living      0.702035
grade            0.667434
sqft_above       0.605567
sqft_living15    0.585379
bathrooms        0.525138
view             0.397293
sqft_basement    0.323816
bedrooms         0.308350
waterfront       0.266369
floors           0.256794
yr_renovated     0.126434
sqft_lot         0.089661
sqft_lot15       0.082447
yr_built         0.054012
condition        0.036362
Name: price, dtype: float64

### Mutual information - another alternative

In [ ]:
# fit the mutual information algorithm
# => that's why either or works
mi = mutual_info_regression(X, y)

# convert results into DataFrame
mi_results = pd.Series(mi, index=X.columns).sort_values(ascending=False)

# high value => variable is strongly connected to target variable
# low value => weak connection
# e.g. very similar to phik-matrix
mi_results

sqft_living      0.350816
grade            0.345337
sqft_living15    0.271310
sqft_above       0.262286
bathrooms        0.210480
sqft_lot15       0.081384
yr_built         0.074706
floors           0.074537
bedrooms         0.074097
sqft_basement    0.070459
sqft_lot         0.059859
view             0.054925
waterfront       0.015225
condition        0.012376
yr_renovated     0.004862
dtype: float64